# NanoFlex - Flexible Pavement Design & ACR/PCR

Python reimplementation of FAARFIELD 2.1.1 flexible pavement thickness design and ACR/PCR computations.

---

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ensure NanoFlex modules are importable
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from materials import MATERIALS, get_default_stack, cbr_to_modulus, modulus_to_cbr, modulus_to_k_value
from structures import PavementSection, PavementLayer, TrafficAircraft
from aircraft import load_aircraft_library, find_aircraft
from design_flex import design_flex, compute_life, DesignResult
from cdf import SubgradeDamageModel
from acr_pcr import compute_acr, ACRResult
from units import UnitSystem

# Global unit system (toggled by the UI radio button below)
usys = UnitSystem(metric=False)

print('NanoFlex modules loaded successfully.')

## 1. Pavement Structure Definition

Select an analysis type and customise the layer properties.

In [ ]:
# ── Unit system toggle ─────────────────────────────────────────────────
w_units = widgets.RadioButtons(
    options=['US Customary', 'Metric (SI)'],
    value='US Customary',
    description='Units:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='250px'),
)

def on_unit_change(change):
    global usys
    usys = UnitSystem(metric=(change['new'] == 'Metric (SI)'))
    build_layer_ui()

w_units.observe(on_unit_change, names='value')

# ── Analysis type selector ─────────────────────────────────────────────
analysis_type_dd = widgets.Dropdown(
    options=['New Flexible', 'HMA on Aggregate', 'HMA Overlay on Flexible'],
    value='New Flexible',
    description='Analysis:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='350px'),
)

# ── Dynamic layer editor ───────────────────────────────────────────────
layer_box = widgets.VBox()
layer_widgets = []   # list of dicts: {name, thickness, modulus, poisson}
current_section = PavementSection(name='Design Section', design_life=20)

def build_layer_ui(change=None):
    global layer_widgets, current_section
    stack = get_default_stack(analysis_type_dd.value)
    layer_widgets = []
    rows = []
    for i, ls in enumerate(stack):
        mat = ls.material
        is_subgrade = mat.layer_code == 4
        w_name = widgets.Label(value=ls.material_name, layout=widgets.Layout(width='260px'))
        t_display = round(usys.display_thickness(ls.thickness), 2) if ls.thickness > 0 else 0.0
        e_display = round(usys.display_modulus(ls.modulus), 2)
        w_thick = widgets.FloatText(
            value=t_display, description=f't ({usys.thickness_label}):', disabled=is_subgrade,
            layout=widgets.Layout(width='150px'),
            style={'description_width': '55px'},
        )
        w_mod = widgets.FloatText(
            value=e_display, description=f'E ({usys.modulus_label}):',
            disabled=not mat.modulus_editable,
            layout=widgets.Layout(width='180px'),
            style={'description_width': '65px'},
        )
        w_poi = widgets.FloatText(
            value=mat.default_poisson, description='nu:',
            layout=widgets.Layout(width='120px'),
            style={'description_width': '30px'},
        )
        layer_widgets.append(dict(name=ls.material_name, thickness=w_thick,
                                  modulus=w_mod, poisson=w_poi))
        rows.append(widgets.HBox([widgets.Label(f'L{i+1}:', layout=widgets.Layout(width='30px')),
                                  w_name, w_thick, w_mod, w_poi]))

    # CBR input for subgrade
    sg_mod = stack[-1].modulus
    cbr_val = modulus_to_cbr(sg_mod)
    w_cbr = widgets.FloatText(value=round(cbr_val, 2), description='CBR:',
                              layout=widgets.Layout(width='150px'),
                              style={'description_width': '40px'})
    def on_cbr_change(change):
        new_e = cbr_to_modulus(change['new'])
        layer_widgets[-1]['modulus'].value = new_e
    w_cbr.observe(on_cbr_change, names='value')
    rows.append(widgets.HBox([widgets.Label('Subgrade CBR:', layout=widgets.Layout(width='120px')),
                              w_cbr]))

    layer_box.children = rows

analysis_type_dd.observe(build_layer_ui, names='value')
build_layer_ui()

# ── Design life ─────────────────────────────────────────────────────────
w_life = widgets.IntText(value=20, description='Design Life (yr):',
                          style={'description_width': '120px'},
                          layout=widgets.Layout(width='200px'))

display(widgets.VBox([
    widgets.HTML('<h3>Pavement Structure</h3>'),
    w_units,
    analysis_type_dd,
    layer_box,
    w_life,
]))

## 2. Traffic Mix

Add aircraft from the FAA library or define custom gear configurations.

In [ ]:
# ── Aircraft library ────────────────────────────────────────────────────
try:
    ac_library = load_aircraft_library()
    ac_names = [f'{r.manufacturer} {r.name}' for r in ac_library]
except Exception:
    ac_library = []
    ac_names = ['(library not found)']

traffic_list = []  # list of TrafficAircraft
traffic_output = widgets.Output()

w_ac_search = widgets.Text(description='Search:', placeholder='e.g. B737, A320',
                           layout=widgets.Layout(width='300px'),
                           style={'description_width': '60px'})
w_ac_select = widgets.Select(options=ac_names[:20], rows=6,
                             layout=widgets.Layout(width='350px'))
w_departures = widgets.IntText(value=1200, description='Dep/yr:',
                               layout=widgets.Layout(width='180px'),
                               style={'description_width': '60px'})
w_growth = widgets.FloatText(value=0.0, description='Growth %:',
                             layout=widgets.Layout(width='180px'),
                             style={'description_width': '70px'})
w_gw_override = widgets.FloatText(value=0, description='GW (lbs):',
                                  layout=widgets.Layout(width='200px'),
                                  style={'description_width': '70px'})

def on_search(change):
    q = change['new']
    if q:
        hits = find_aircraft(ac_library, q)
        w_ac_select.options = [f'{r.manufacturer} {r.name}' for r in hits]
    else:
        w_ac_select.options = ac_names[:20]

w_ac_search.observe(on_search, names='value')

def add_aircraft(_):
    sel = w_ac_select.value
    if not sel:
        return
    rec = None
    for r in ac_library:
        if f'{r.manufacturer} {r.name}' == sel:
            rec = r
            break
    if rec is None:
        return
    gw = w_gw_override.value if w_gw_override.value > 0 else rec.gross_weight_lbs
    tac = rec.to_traffic_aircraft(
        annual_departures=w_departures.value,
        annual_growth=w_growth.value,
    )
    tac.gross_weight = gw
    traffic_list.append(tac)
    refresh_traffic_table()

def clear_traffic(_):
    traffic_list.clear()
    refresh_traffic_table()

def refresh_traffic_table():
    with traffic_output:
        clear_output()
        if not traffic_list:
            print('No aircraft in traffic mix.')
            return
        gw_lbl = f'GW ({usys.weight_label})'
        p_lbl = f'p ({usys.pressure_label})'
        print(f'{"#":>2}  {"Name":<25} {gw_lbl:>12} {"Dep/yr":>8} {"Wheels":>6} {p_lbl:>10} {"mg%":>5}')
        print('-' * 80)
        for i, ac in enumerate(traffic_list):
            gw_d = usys.display_weight(ac.gross_weight)
            p_d = usys.display_pressure(ac.tire_pressure)
            print(f'{i+1:>2}  {ac.name:<25} {gw_d:>12,.0f} {ac.annual_departures:>8} '
                  f'{ac.n_wheels:>6} {p_d:>10.0f} {ac.mg_percent:>5.2f}')

btn_add = widgets.Button(description='Add Aircraft', button_style='success',
                          layout=widgets.Layout(width='120px'))
btn_clear = widgets.Button(description='Clear All', button_style='danger',
                            layout=widgets.Layout(width='100px'))
btn_add.on_click(add_aircraft)
btn_clear.on_click(clear_traffic)

refresh_traffic_table()

display(widgets.VBox([
    widgets.HTML('<h3>Traffic Mix</h3>'),
    widgets.HBox([w_ac_search, w_ac_select]),
    widgets.HBox([w_departures, w_growth, w_gw_override]),
    widgets.HBox([btn_add, btn_clear]),
    traffic_output,
]))

## 3. Run Flexible Thickness Design

Click **Design** to iterate the structure to CDF = 1.0.

In [ ]:
design_output = widgets.Output()

def build_section():
    """Build PavementSection from current UI state, converting to US Customary."""
    section = PavementSection(name='Design', design_life=w_life.value)
    for lw in layer_widgets:
        section.layers.append(PavementLayer(
            material_name=lw['name'],
            thickness=usys.to_internal_thickness(lw['thickness'].value),
            modulus=usys.to_internal_modulus(lw['modulus'].value),
            poisson=lw['poisson'].value,
        ))
    section.traffic = list(traffic_list)
    # Validate before running
    warnings_list = section.validate()
    if warnings_list:
        print('Validation warnings:')
        for w in warnings_list:
            print(f'  - {w}')
        print()
    return section

def run_design(_):
    with design_output:
        clear_output()
        if not traffic_list:
            print('Add at least one aircraft to the traffic mix.')
            return
        section = build_section()
        print('Running flexible thickness design...\n')
        result = design_flex(section, verbose=True)

        print(f'\n{"=" * 60}')
        print(f'  Status: {result.message}')
        print(f'  Iterations: {result.iterations}')
        print(f'  CDF (subgrade): {result.cdf_subgrade:.6f}')
        if result.cdf_asphalt is not None:
            print(f'  CDF (asphalt):  {result.cdf_asphalt:.6f}')
        t_lbl = usys.thickness_label
        e_lbl = usys.modulus_label
        print(f'\n  Final Structure:')
        print(f'  {"Layer":<5} {"Material":<35} {"t ("+t_lbl+")":>8} {"E ("+e_lbl+")":>12}')
        print(f'  {"-"*65}')
        for i, (t, lw) in enumerate(zip(result.layer_thicknesses, layer_widgets)):
            t_disp = usys.display_thickness(t)
            e_disp = usys.display_modulus(usys.to_internal_modulus(lw["modulus"].value))
            print(f'  {i+1:<5} {lw["name"]:<35} {t_disp:>8.2f} {e_disp:>12,.1f}')
            lw['thickness'].value = round(usys.display_thickness(t), 2)
        total = sum(usys.display_thickness(t) for t in result.layer_thicknesses if t > 0)
        print(f'  {"":5} {"TOTAL":35} {total:>8.2f}')

btn_design = widgets.Button(description='Design', button_style='primary',
                             icon='cogs', layout=widgets.Layout(width='150px'))
btn_design.on_click(run_design)

display(widgets.VBox([btn_design, design_output]))

## 4. Compute ACR

Compute Aircraft Classification Rating for each aircraft in the traffic mix.

In [ ]:
acr_output = widgets.Output()

w_acr_cats = widgets.SelectMultiple(
    options=['A', 'B', 'C', 'D'], value=['D'],
    description='Categories:', rows=4,
    style={'description_width': '90px'},
    layout=widgets.Layout(width='200px'),
)

def run_acr(_):
    with acr_output:
        clear_output()
        if not traffic_list:
            print('Add at least one aircraft first.')
            return
        cats = list(w_acr_cats.value)
        print(f'Computing ACR for {len(traffic_list)} aircraft, categories: {cats}\n')
        for ac in traffic_list:
            print(f'--- {ac.name} (GW={ac.gross_weight:,.0f} lbs) ---')
            result = compute_acr(ac, categories=cats, verbose=False)
            for cat in cats:
                print(f'  ACR({cat}) = {result.acr[cat]:>8.1f}  '
                      f'DSWL = {result.dswl_lbs[cat]:>10,.0f} lbs  '
                      f'Ref t = {result.reference_thickness[cat]:>6.1f} in')
            print()

btn_acr = widgets.Button(description='Compute ACR', button_style='warning',
                          icon='calculator', layout=widgets.Layout(width='160px'))
btn_acr.on_click(run_acr)

display(widgets.VBox([
    widgets.HBox([btn_acr, w_acr_cats]),
    acr_output,
]))

## 5. Manual Analysis

For scripting and custom computations, you can use the modules directly.

In [ ]:
# Example: Manual flexible design
#
# section = PavementSection(name='Manual', design_life=20)
# section.layers = [
#     PavementLayer('P-401/P-403 HMA Surface', 4.0, 200000),
#     PavementLayer('P-209 Crushed Aggregate', 10.0, 75000),
#     PavementLayer('Subgrade', 0.0, 15000),
# ]
# section.traffic = [TrafficAircraft(
#     name='B737-800', gross_weight=174200, mg_percent=0.95,
#     tire_pressure=204, n_wheels=2,
#     wheel_x=[-8.5, 8.5], wheel_y=[0.0, 0.0],
#     eval_x=[0.0], eval_y=[0.0],
#     annual_departures=3000,
# )]
# result = design_flex(section, verbose=True)
# print(f'Design thickness: {result.layer_thicknesses}')